In [ ]:
import math
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Dokumen singkat topik teknik komputer
dokumen_teknis = [
    "pemasangan server Linux dan konfigurasi server jaringan",
    "kompetisi pemrograman web menggunakan bahasa Python",
    "pelatihan instalasi jaringan kabel optik dan server",
    "pengolahan sinyal digital pada sistem jaringan"
]

# Preprocessing sederhana: case folding, tokenisasi, dan hapus stopword
stopwords_sederhana = {'dan', 'pada', 'menggunakan'}

def preprocess_simpel(teks):
    kata = teks.lower().split()
    return [k for k in kata if k not in stopwords_sederhana]

# Proses semua dokumen
docs_clean = [preprocess_simpel(doc) for doc in dokumen_teknis]

# Ambil daftar kata unik (vocabulary)
vocab = sorted(list(set(kata for doc in docs_clean for kata in doc)))
total_dokumen = len(dokumen_teknis)

print("Vocabulary:", vocab)

# --- PERHITUNGAN MANUAL ---

# 1. Hitung Term Frequency (TF - Raw Count)
matrix_tf = []
for doc in docs_clean:
    row = [doc.count(term) for term in vocab]
    matrix_tf.append(row)

df_tf = pd.DataFrame(matrix_tf, columns=vocab, index=[f'Dokumen {i+1}' for i in range(total_dokumen)])

# 2. Hitung Document Frequency (DF) & Inverse Document Frequency (IDF)
# Rumus Smooth IDF: ln((1 + N) / (1 + df)) + 1
df_counts = {term: sum(1 for doc in docs_clean if term in doc) for term in vocab}
idf_values = {term: math.log((1 + total_dokumen) / (1 + df)) + 1 for term, df in df_counts.items()}

df_idf = pd.DataFrame([df_counts, idf_values], index=['df (Doc Freq)', 'IDF Score'], columns=vocab)

# 3. Hitung TF-IDF Manual dan Normalisasi L2
tfidf_manual_matrix = []
for row_tf in matrix_tf:
    raw_tfidf = [row_tf[i] * idf_values[vocab[i]] for i in range(len(vocab))]
    norm = math.sqrt(sum(x**2 for x in raw_tfidf))
    norm_tfidf = [round(x / norm, 4) if norm > 0 else 0 for x in raw_tfidf]
    tfidf_manual_matrix.append(norm_tfidf)

df_tfidf_manual = pd.DataFrame(tfidf_manual_matrix, columns=vocab, index=[f'Dokumen {i+1}' for i in range(total_dokumen)])

# Perhitungan scikit learn
docs_joined = [' '.join(doc) for doc in docs_clean]
vectorizer = TfidfVectorizer()
tfidf_sklearn = vectorizer.fit_transform(docs_joined)

df_tfidf_sklearn = pd.DataFrame(np.round(tfidf_sklearn.toarray(), 4),
                                columns=vectorizer.get_feature_names_out(),
                                index=[f'Dokumen {i+1}' for i in range(total_dokumen)])

# Menampilkan hasil
print("\n" + "="*50)
print("1. TABEL BAG-OF-WORDS / TERM FREQUENCY (TF)")
print("="*50)
print(df_tf)

print("\n" + "="*50)
print("2. NILAI DF DAN IDF KATA")
print("="*50)
print(df_idf.round(4))

print("\n" + "="*50)
print("3. MATRIKS TF-IDF (PERHITUNGAN MANUAL)")
print("="*50)
print(df_tfidf_manual)

print("\n" + "="*50)
print("4. MATRIKS TF-IDF (SCIKIT-LEARN)")
print("="*50)
print(df_tfidf_sklearn)

Vocabulary: ['bahasa', 'digital', 'instalasi', 'jaringan', 'kabel', 'kompetisi', 'konfigurasi', 'linux', 'optik', 'pelatihan', 'pemasangan', 'pemrograman', 'pengolahan', 'python', 'server', 'sinyal', 'sistem', 'web']

1. TABEL BAG-OF-WORDS / TERM FREQUENCY (TF)
           bahasa  digital  instalasi  jaringan  kabel  kompetisi  \
Dokumen 1       0        0          0         1      0          0   
Dokumen 2       1        0          0         0      0          1   
Dokumen 3       0        0          1         1      1          0   
Dokumen 4       0        1          0         1      0          0   

           konfigurasi  linux  optik  pelatihan  pemasangan  pemrograman  \
Dokumen 1            1      1      0          0           1            0   
Dokumen 2            0      0      0          0           0            1   
Dokumen 3            0      0      1          1           0            0   
Dokumen 4            0      0      0          0           0            0   

           

Analisis:

1. Dokumen 1: Kata "server" memiliki bobot paling tinggi karena muncul dua kali di dalam teks (raw count = 2).
2. Dokumen 2: Kata "bahasa", "kompetisi", "pemrograman", "python", dan "web" mendapat nilai paling tinggi karena kata-kata ini langka dan cuma muncul di dokumen ini saja.   
3. Dokumen 3: Kata "instalasi", "kabel", "optik", dan "pelatihan" dominan karena sifatnya spesifik pada dokumen tersebut.
4. Dokumen 4: Kata "digital", "pengolahan", dan "sinyal" mencatatkan bobot paling tinggi karena hanya ada di dokumen keempat.

Mengapa Term Tersebut Penting?

Semakin langka suatu kata di dokumen lain, skor IDF-nya akan semakin tinggi. Kata yang unik dan spesifik inilah yang membantu mesin pencari mengenali topik utama dari suatu dokumen.